# 10 Cross-Validation Results Output

Version 3: Table 9 includes **Business Priority Rank** and **Business Priority Score**, replacing Scheme 6 Theoretical Priority Score.

**North Star:** One result → One table → One bar graph.

In [2]:

# ============================================================
# 10 Cross-Validation Results Output - Publication Tables & Bar Charts
# Version 3: Table 9 includes Business Priority Rank and Business Priority Score
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x

# ============================================================
# 1. Configuration
# ============================================================

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")
    except Exception as e:
        print("Google Drive mount skipped:", e)
        ROOT_PATH = Path(".")
else:
    ROOT_PATH = Path(".")

RESULTS_ROOT = ROOT_PATH / "07_Results"
INPUT_09_TABLES_PATH = RESULTS_ROOT / "09_cross_dataset_framework_validation" / "tables"
OUTPUT_ROOT = RESULTS_ROOT / "10_cross_validation_results_output_framework_validation"
FIGURES_PATH = OUTPUT_ROOT / "publication_outputs" / "figures"
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

DPI = 600
ROUND_DECIMALS = 3

plt.rcParams.update({
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

# ============================================================
# 2. Helper Functions
# ============================================================

def read_table(filename, required=True):
    path = INPUT_09_TABLES_PATH / filename
    if not path.exists():
        msg = f"Missing input table: {path}"
        if required:
            raise FileNotFoundError(msg)
        print(msg)
        return pd.DataFrame()
    return pd.read_csv(path)

def clean_columns(df):
    out = df.copy()
    out.columns = (
        out.columns.str.strip()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.lower()
    )
    return out

def title_case_columns(df):
    out = df.copy()
    out.columns = [c.replace("_", " ").title() for c in out.columns]
    return out

def round_numeric(df, decimals=ROUND_DECIMALS):
    out = df.copy()
    numeric_cols = out.select_dtypes(include=[np.number]).columns
    out[numeric_cols] = out[numeric_cols].round(decimals)
    return out

def show_table(title, df, max_rows=100):
    display(Markdown(f"### {title}"))
    if df.empty:
        print("No data available.")
        return
    display(title_case_columns(round_numeric(df)).head(max_rows))

def show_ranking_table(title, df, rank_col, dataset_col="dataset_label"):
    display(Markdown(f"### {title}"))
    if df.empty:
        print("No data available.")
        return
    if dataset_col in df.columns:
        for dataset in df[dataset_col].dropna().unique():
            sub = df[df[dataset_col] == dataset].copy()
            if rank_col in sub.columns:
                sub = sub.sort_values(rank_col)
            display(Markdown(f"#### {dataset}"))
            display(title_case_columns(round_numeric(sub.drop(columns=[dataset_col], errors="ignore"))))
    else:
        sub = df.copy()
        if rank_col in sub.columns:
            sub = sub.sort_values(rank_col)
        display(title_case_columns(round_numeric(sub)))

def add_bar_labels(ax, orientation="vertical", decimals=3, fontsize=8):
    for container in ax.containers:
        labels = []
        for value in container.datavalues:
            if pd.isna(value):
                labels.append("")
            elif abs(value) < 10 and decimals > 0:
                labels.append(f"{value:.{decimals}f}")
            else:
                labels.append(f"{value:.0f}")
        if orientation == "horizontal":
            ax.bar_label(container, labels=labels, label_type="edge", padding=3, fontsize=fontsize)
        else:
            ax.bar_label(container, labels=labels, label_type="edge", padding=2, fontsize=fontsize)

def place_legend(ax, ncol=3):
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=ncol, frameon=False)

def save_bar_chart(df, x_col, y_col, title, filename, hue_col=None,
                   xlabel=None, ylabel=None, sort_by=None, ascending=False,
                   horizontal=False, figsize=(9,5), decimals=3):
    if df.empty:
        print(f"Skipped figure because dataframe is empty: {filename}")
        return None

    data = df.copy()
    if sort_by is not None and hue_col is None:
        data = data.sort_values(sort_by, ascending=ascending)

    fig, ax = plt.subplots(figsize=figsize)

    if hue_col is None:
        labels = data[x_col].astype(str).tolist()
        values = data[y_col].astype(float).tolist()
        if horizontal:
            ax.barh(labels, values)
            ax.invert_yaxis()
            ax.set_xlabel(ylabel or y_col.replace("_", " ").title())
            ax.set_ylabel(xlabel or "")
            add_bar_labels(ax, "horizontal", decimals)
        else:
            ax.bar(labels, values)
            ax.set_ylabel(ylabel or y_col.replace("_", " ").title())
            ax.set_xlabel(xlabel or "")
            ax.tick_params(axis="x", rotation=35)
            add_bar_labels(ax, "vertical", decimals)
    else:
        pivot = data.pivot_table(index=x_col, columns=hue_col, values=y_col, aggfunc="mean")
        pivot.plot(kind="bar", ax=ax)
        ax.set_ylabel(ylabel or y_col.replace("_", " ").title())
        ax.set_xlabel(xlabel or "")
        ax.tick_params(axis="x", rotation=35)
        add_bar_labels(ax, "vertical", decimals, fontsize=7)
        place_legend(ax, ncol=min(3, len(pivot.columns)))

    ax.set_title(title, pad=12)
    ax.grid(axis="y", alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if not horizontal and y_col in data.columns:
        ymax = data[y_col].max()
        if pd.notna(ymax):
            ax.set_ylim(0, max(1.05, ymax * 1.18))

    fig.tight_layout()
    output_path = FIGURES_PATH / filename
    fig.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved PNG: {output_path}")
    return output_path

def save_grouped_rank_chart(df, dataset_col, model_col, rank_cols, title, filename):
    if df.empty:
        print(f"Skipped figure because dataframe is empty: {filename}")
        return None
    data = df.copy()
    data["dataset_model"] = data[dataset_col].astype(str) + " | " + data[model_col].astype(str)
    plot_df = data[["dataset_model"] + rank_cols].set_index("dataset_model")
    fig, ax = plt.subplots(figsize=(11, 6))
    plot_df.plot(kind="bar", ax=ax)
    ax.set_title(title, pad=12)
    ax.set_ylabel("Rank (lower is better)")
    ax.set_xlabel("")
    ax.invert_yaxis()
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=70)
    add_bar_labels(ax, "vertical", decimals=0, fontsize=7)
    place_legend(ax, ncol=2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    output_path = FIGURES_PATH / filename
    fig.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved PNG: {output_path}")
    return output_path

def find_first_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

def pick_cols(df, cols):
    return [c for c in cols if c in df.columns]

# ============================================================
# 3. Load Final Outputs from Notebook 09
# ============================================================

dataset_overview_df = clean_columns(read_table("dataset_overview.csv"))
baseline_df = clean_columns(read_table("baseline_component_comparison.csv"))
fidelity_df = clean_columns(read_table("fidelity_component_comparison.csv"))
stability_df = clean_columns(read_table("stability_component_comparison.csv"))
actionability_df = clean_columns(read_table("actionability_component_comparison.csv"))
regulatory_df = clean_columns(read_table("regulatory_component_comparison.csv"))
ees_df = clean_columns(read_table("ees_component_comparison.csv"))
average_ees_df = clean_columns(read_table("cross_dataset_average_ees.csv"))
ahp_df = clean_columns(read_table("ahp_scheme_summary.csv", required=False))
robustness_df = clean_columns(read_table("ranking_robustness_comparison.csv", required=False))
winner_matrix_df = clean_columns(read_table("cross_dataset_winner_matrix.csv"))

print("Loaded Notebook 09 tables from:", INPUT_09_TABLES_PATH)
print("Figures will be saved to:", FIGURES_PATH)

# ============================================================
# 4. Tables and Figures
# ============================================================

show_table("Table 1. Characteristics of the Three High-Stakes Validation Domains", dataset_overview_df)

models_table_df = pd.DataFrame([
    {"model_name": "Logistic Regression", "model_type": "Interpretable linear model", "role": "Baseline interpretable classifier"},
    {"model_name": "Decision Tree", "model_type": "Interpretable tree model", "role": "Rule-based baseline classifier"},
    {"model_name": "Random Forest", "model_type": "Ensemble model", "role": "Black-box benchmark"},
    {"model_name": "XGBoost", "model_type": "Gradient boosting ensemble", "role": "High-performance black-box benchmark"},
    {"model_name": "MLP", "model_type": "Neural network", "role": "Nonlinear black-box benchmark"},
])
show_table("Table 2. Machine Learning Models Evaluated", models_table_df)

table3_df = baseline_df[pick_cols(baseline_df, [
    "dataset_label", "business_priority_rank", "model_name",
    "recall", "roc_auc", "precision", "accuracy", "business_priority_score"
])].copy()
show_ranking_table("Table 3. Baseline Predictive Performance and Business Priority Score", table3_df, "business_priority_rank")
save_bar_chart(baseline_df, "model_name", "business_priority_score",
               "Baseline Business Priority Score Across Validation Domains",
               "Figure_02A_Baseline_Business_Priority_Score.png",
               hue_col="dataset_label", xlabel="Model", ylabel="Business Priority Score")

baseline_rank_col = find_first_column(baseline_df, ["business_priority_rank", "baseline_rank"])
ees_rank_col = find_first_column(ees_df, ["ees_rank", "rank"])
if baseline_rank_col and ees_rank_col:
    rank_compare_df = (
        baseline_df[["dataset_label", "model_name", baseline_rank_col]]
        .merge(ees_df[["dataset_label", "model_name", ees_rank_col]],
               on=["dataset_label", "model_name"], how="inner")
        .rename(columns={baseline_rank_col: "Baseline Rank", ees_rank_col: "EES Rank"})
    )
    show_ranking_table("Figure 2B Source Table. Baseline Rank vs Final EES Rank", rank_compare_df, "EES Rank")
    save_grouped_rank_chart(rank_compare_df, "dataset_label", "model_name",
                            ["Baseline Rank", "EES Rank"],
                            "Baseline Predictive Rank vs Final EES Rank",
                            "Figure_02B_Baseline_Rank_vs_EES_Rank.png")

table5_df = fidelity_df[pick_cols(fidelity_df, [
    "dataset_label", "fidelity_rank", "model_name",
    "shap_reconstruction_fidelity_score", "lime_fidelity_score",
    "shap_lime_agreement_score", "fidelity_score"
])]
show_ranking_table("Table 5. Fidelity Evaluation Across the Three Validation Domains", table5_df, "fidelity_rank")
save_bar_chart(fidelity_df, "model_name", "fidelity_score",
               "Composite Fidelity Score Across Validation Domains",
               "Figure_03A_Composite_Fidelity_Score.png",
               hue_col="dataset_label", xlabel="Model", ylabel="Fidelity Score")

fidelity_component_cols = pick_cols(fidelity_df, [
    "shap_reconstruction_fidelity_score", "lime_fidelity_score", "shap_lime_agreement_score"
])
if fidelity_component_cols:
    fidelity_long_df = fidelity_df.melt(
        id_vars=["dataset_label", "model_name"],
        value_vars=fidelity_component_cols,
        var_name="component", value_name="score"
    )
    fidelity_long_df["component"] = (
        fidelity_long_df["component"].str.replace("_score", "", regex=False)
        .str.replace("_", " ").str.title()
    )
    save_bar_chart(fidelity_long_df, "component", "score",
                   "Component Contributions to the Fidelity Pillar",
                   "Figure_03B_Fidelity_Component_Contributions.png",
                   hue_col="dataset_label", xlabel="Fidelity Component", ylabel="Score",
                   figsize=(10,5))

table6_df = stability_df[pick_cols(stability_df, [
    "dataset_label", "stability_rank", "model_name", "mean_local_lipschitz_ratio", "stability_score"
])]
show_ranking_table("Table 6. Stability Evaluation Across the Three Validation Domains", table6_df, "stability_rank")
save_bar_chart(stability_df, "model_name", "stability_score",
               "Stability Score Across Validation Domains",
               "Figure_04_Stability_Score.png",
               hue_col="dataset_label", xlabel="Model", ylabel="Stability Score")

table7_df = actionability_df[pick_cols(actionability_df, [
    "dataset_label", "actionability_rank", "model_name",
    "availability_score", "proximity_score", "sparsity_score", "actionability_score"
])]
show_ranking_table("Table 7. Counterfactual Actionability Evaluation Across the Three Validation Domains", table7_df, "actionability_rank")
save_bar_chart(actionability_df, "model_name", "actionability_score",
               "Counterfactual Actionability Score Across Validation Domains",
               "Figure_05_Actionability_Score.png",
               hue_col="dataset_label", xlabel="Model", ylabel="Actionability Score")

table8_df = regulatory_df[pick_cols(regulatory_df, [
    "dataset_label", "regulatory_alignment_rank", "rank", "model_name",
    "transparency_score", "explanation_availability_score",
    "explanation_reliability_evidence", "explanation_robustness_evidence", "contestability_evidence",
    "fidelity_alignment_score", "stability_alignment_score", "actionability_contestability_score",
    "sensitive_feature_awareness_score", "documentation_auditability_score",
    "regulatory_alignment_score", "regulatory_alignment_band"
])]
rank_col = "regulatory_alignment_rank" if "regulatory_alignment_rank" in table8_df.columns else "rank"
show_ranking_table("Table 8. Regulatory Alignment Evaluation Across the Three Validation Domains", table8_df, rank_col)
save_bar_chart(regulatory_df, "model_name", "regulatory_alignment_score",
               "Regulatory Alignment Score Across Validation Domains",
               "Figure_06_Regulatory_Alignment_Score.png",
               hue_col="dataset_label", xlabel="Model", ylabel="Regulatory Alignment Score")

# ============================================================
# Table 9 Update:
# Final EES table includes Business Priority Rank and Business Priority Score.
# Scheme 6 Theoretical Priority Score is intentionally excluded.
# ============================================================

table9_df = (
    ees_df
    .merge(
        baseline_df[["dataset_label", "model_name", "business_priority_rank", "business_priority_score"]],
        on=["dataset_label", "model_name"],
        how="left"
    )
)
table9_df = table9_df[pick_cols(table9_df, [
    "dataset_label",
    "ees_rank",
    "business_priority_rank",
    "model_name",
    "fidelity_score",
    "stability_score",
    "actionability_score",
    "regulatory_alignment_score",
    "ees_score",
    "business_priority_score",
    "ees_band"
])].copy()
show_ranking_table(
    "Table 9. Final Explainability-Ethics Score (EES) Rankings Across the Three Validation Domains",
    table9_df,
    "ees_rank"
)

save_bar_chart(ees_df, "model_name", "ees_score",
               "Final Explainability-Ethics Score Across Validation Domains",
               "Figure_07_Final_EES_Score.png",
               hue_col="dataset_label", xlabel="Model", ylabel="EES Score")

table10_df = average_ees_df[pick_cols(average_ees_df, [
    "cross_dataset_ees_rank", "model_name", "mean_ees_score", "median_ees_score",
    "min_ees_score", "max_ees_score", "n_datasets"
])]
show_ranking_table("Table 10. Average Explainability-Ethics Score Across the Three Validation Domains",
                   table10_df, "cross_dataset_ees_rank")
save_bar_chart(average_ees_df, "model_name", "mean_ees_score",
               "Average Explainability-Ethics Score Across Validation Domains",
               "Figure_08_Average_EES_Score.png",
               xlabel="Model", ylabel="Mean EES Score",
               sort_by="mean_ees_score", ascending=False, horizontal=True)

if not ahp_df.empty:
    table11_df = ahp_df[pick_cols(ahp_df, [
        "dataset_name", "dataset_label", "domain", "scheme_id",
        "scheme_name", "scheme_role", "lambda_max",
        "consistency_index", "random_index", "consistency_ratio",
        "is_consistent_under_0_10"
    ])]
    show_table("Table 11. AHP Consistency Diagnostics", table11_df, max_rows=100)
    if "consistency_ratio" in ahp_df.columns:
        save_bar_chart(ahp_df, "scheme_name", "consistency_ratio",
                       "AHP Consistency Ratio Across Weighting Schemes",
                       "Figure_09_AHP_Consistency_Ratio.png",
                       hue_col="dataset_label" if "dataset_label" in ahp_df.columns else None,
                       xlabel="AHP Scheme", ylabel="Consistency Ratio",
                       figsize=(11,5), decimals=3)

if not robustness_df.empty:
    show_table("Optional Source Table. Ranking Robustness Comparison", robustness_df, max_rows=100)
    robustness_y = find_first_column(robustness_df, [
        "max_rank_change_vs_equal_weight", "rank_change_vs_equal_weight", "ees_score", "score"
    ])
    robustness_x = find_first_column(robustness_df, ["model_name", "scheme_name", "dataset_label"])
    if robustness_y and robustness_x:
        save_bar_chart(robustness_df, robustness_x, robustness_y,
                       "AHP Ranking Robustness",
                       "Figure_10_AHP_Ranking_Robustness.png",
                       hue_col="dataset_label" if "dataset_label" in robustness_df.columns else None,
                       xlabel=robustness_x.replace("_", " ").title(),
                       ylabel=robustness_y.replace("_", " ").title(),
                       figsize=(10,5))

show_table("Table 12. Cross-Dataset Winner Matrix", winner_matrix_df)
if not winner_matrix_df.empty:
    id_col = "section" if "section" in winner_matrix_df.columns else winner_matrix_df.columns[0]
    winner_long_df = winner_matrix_df.melt(id_vars=[id_col], var_name="dataset_label", value_name="winner_model_name")
    winner_count_df = (
        winner_long_df.dropna(subset=["winner_model_name"])
        .groupby("winner_model_name", as_index=False).size()
        .rename(columns={"size": "n_wins"})
        .sort_values("n_wins", ascending=False)
    )
    show_table("Figure 11 Source Table. Cross-Dataset Winner Counts", winner_count_df)
    save_bar_chart(winner_count_df, "winner_model_name", "n_wins",
                   "Cross-Dataset Winner Counts",
                   "Figure_11_Cross_Dataset_Winner_Counts.png",
                   xlabel="Model", ylabel="Number of Wins",
                   sort_by="n_wins", ascending=False, horizontal=True, decimals=0)

print("=" * 100)
print("Notebook 10 completed successfully.")
print(f"Input tables from Notebook 09: {INPUT_09_TABLES_PATH}")
print(f"Publication PNG figures saved to: {FIGURES_PATH}")
print("=" * 100)


Output hidden; open in https://colab.research.google.com to view.